## 0. 환경 설정

In [1]:
import os
import sys
import glob
import shutil
import numpy as np
from PIL import Image

# DUSt3R 저장소 클론 (이미 존재하면 건너뜀)
if not os.path.exists("/content/dust3r"):
    !git clone --recursive https://github.com/naver/dust3r.git /content/dust3r

%cd /content/dust3r
!pip install -q -r requirements.txt
!pip install -q open3d shapely trimesh scipy roma
%cd /content

# DUSt3R 모듈 경로 등록 (중복 방지)
if "/content/dust3r" not in sys.path:
    sys.path.insert(0, "/content/dust3r")

Cloning into '/content/dust3r'...
remote: Enumerating objects: 611, done.
remote: Total 611 (delta 0), reused 0 (delta 0), pack-reused 611 (from 1)
Receiving objects: 100% (611/611), 756.60 KiB | 39.82 MiB/s, done.
Resolving deltas: 100% (355/355), done.
Submodule 'croco' (https://github.com/naver/croco) registered for path 'croco'
Cloning into '/content/dust3r/croco'...
remote: Enumerating objects: 198, done.        
remote: Counting objects: 100% (87/87), done.        
remote: Compressing objects: 100% (54/54), done.        
remote: Total 198 (delta 54), reused 33 (delta 33), pack-reused 111 (from 1)        
Receiving objects: 100% (198/198), 403.93 KiB | 8.98 MiB/s, done.
Resolving deltas: 100% (94/94), done.
Submodule path 'croco': checked out 'd7de0705845239092414480bd829228723bf20de'
/content/dust3r
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ===== 1. 영상에서 10 or 100장 균등 추출 =====
import cv2, os
from pathlib import Path

VIDEO_PATH = "/content/room.mov"
FRAMES_DIR = "/content/frames"
N_FRAMES = 30

os.makedirs(FRAMES_DIR, exist_ok=True)
for f in Path(FRAMES_DIR).glob("*.jpg"):
    f.unlink()

cap = cv2.VideoCapture(VIDEO_PATH)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
indices = [int(i * total / N_FRAMES) for i in range(N_FRAMES)]

saved = 0
for idx in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        cv2.imwrite(f"{FRAMES_DIR}/frame_{saved:04d}.jpg", frame)
        saved += 1
cap.release()
print(f" {saved}장 추출 (영상 총 {total}프레임 중 균등 샘플링)")

 30장 추출 (영상 총 1611프레임 중 균등 샘플링)


In [3]:
import torch
from dust3r.model import AsymmetricCroCo3DStereo
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.inference import inference
from dust3r.cloud_opt import global_aligner

# DUSt3R 모델 로드
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AsymmetricCroCo3DStereo.from_pretrained(
    "naver/DUSt3R_ViTLarge_BaseDecoder_512_dpt"
).to(device)
model.eval()
print(f"DUSt3R 모델 로드 완료 (device: {device})")

Warning, cannot find cuda-compiled version of RoPE2D, using a slow pytorch version instead


/content/dust3r/dust3r/cloud_opt/base_opt.py:275: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

DUSt3R 모델 로드 완료 (device: cuda)


In [4]:
# ===== 4. 이미지 로드 + 쌍 구성 + 추론 + Global alignment =====
import glob
from PIL import Image

img_paths = sorted(glob.glob(f"{FRAMES_DIR}/*.jpg"))
images_pil = [Image.open(p).convert('RGB') for p in img_paths]  # 컬러 매핑용 원본

images_dust3r = load_images(img_paths, size=512)
pairs = make_pairs(images_dust3r, scene_graph="complete", prefilter=None, symmetrize=True)
print(f"이미지 쌍 {len(pairs)}개 구성")

with torch.no_grad():
    output = inference(pairs, model, device, batch_size=2)

scene = global_aligner(output, device=device)
scene.compute_global_alignment(niter=300, init="mst")
print("3D 복원 및 정렬 완료")

>> Loading a list of 30 images
 - adding /content/frames/frame_0000.jpg with resolution 1440x1920 --> 384x512
 - adding /content/frames/frame_0001.jpg with resolution 1440x1920 --> 384x512
 - adding /content/frames/frame_0002.jpg with resolution 1440x1920 --> 384x512
 - adding /content/frames/frame_0003.jpg with resolution 1440x1920 --> 384x512
 - adding /content/frames/frame_0004.jpg with resolution 1440x1920 --> 384x512
 - adding /content/frames/frame_0005.jpg with resolution 1440x1920 --> 384x512
 - adding /content/frames/frame_0006.jpg with resolution 1440x1920 --> 384x512
 - adding /content/frames/frame_0007.jpg with resolution 1440x1920 --> 384x512
 - adding /content/frames/frame_0008.jpg with resolution 1440x1920 --> 384x512
 - adding /content/frames/frame_0009.jpg with resolution 1440x1920 --> 384x512
 - adding /content/frames/frame_0010.jpg with resolution 1440x1920 --> 384x512
 - adding /content/frames/frame_0011.jpg with resolution 1440x1920 --> 384x512
 - adding /content/fr

  0%|          | 0/435 [00:00<?, ?it/s]/content/dust3r/dust3r/inference.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(use_amp)):
/content/dust3r/dust3r/model.py:206: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/content/dust3r/dust3r/inference.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 435/435 [01:14<00:00,  5.81it/s]


 init edge (5*,4*) score=np.float64(436.76708984375)
 init edge (6*,5) score=np.float64(371.0095520019531)
 init edge (2*,5) score=np.float64(322.627685546875)
 init edge (0*,2) score=np.float64(317.6027526855469)
 init edge (2,3*) score=np.float64(301.3603515625)
 init edge (28*,2) score=np.float64(293.51611328125)
 init edge (7*,6) score=np.float64(242.32626342773438)
 init edge (1*,2) score=np.float64(337.1406555175781)
 init edge (29*,28) score=np.float64(296.0510559082031)
 init edge (8*,7) score=np.float64(291.2079772949219)
 init edge (29,27*) score=np.float64(285.81365966796875)
 init edge (29,26*) score=np.float64(49.979679107666016)
 init edge (8,9*) score=np.float64(340.3711853027344)
 init edge (10*,9) score=np.float64(371.7572326660156)
 init edge (10,12*) score=np.float64(337.835693359375)
 init edge (10,13*) score=np.float64(265.4771728515625)
 init edge (14*,13) score=np.float64(262.9897766113281)
 init edge (14,16*) score=np.float64(166.63726806640625)
 init edge (16,1

  0%|          | 0/300 [00:00<?, ?it/s]/content/dust3r/dust3r/cloud_opt/base_opt.py:366: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  return float(loss), lr
100%|██████████| 300/300 [01:15<00:00,  3.95it/s, lr=1.27413e-06 loss=0.00851851]

3D 복원 및 정렬 완료


In [5]:
# ===== 5. 포인트 클라우드 추출 — 고정 threshold 방식 =====
import numpy as np

pointmaps = scene.get_pts3d()
confidences = scene.get_conf()

CONF_THR = 1.0

all_pts, all_cols = [], []
for i, pts in enumerate(pointmaps):
    pts_np = pts.detach().cpu().numpy().reshape(-1, 3)
    conf_np = confidences[i].detach().cpu().numpy().flatten()
    mask = conf_np > CONF_THR

    h, w = pts.shape[:2]
    img_resized = np.array(images_pil[i].resize((w, h))).reshape(-1, 3)

    all_pts.append(pts_np[mask])
    all_cols.append(img_resized[mask])

all_pts = np.concatenate(all_pts, axis=0)
all_cols = np.concatenate(all_cols, axis=0)  # 0~255 uint8
print(f"필터링 후 포인트 수: {len(all_pts):,}")

필터링 후 포인트 수: 5,849,021


In [7]:
# ===== 6. GLB 저장 (unlit 머티리얼 적용) =====
!pip install -q trimesh pygltflib
import trimesh
from pygltflib import GLTF2

def to_opengl(points):
    """DUSt3R 좌표계 -> OpenGL/GLB 좌표계 (Y, Z 반전) — 바닥 정렬 이후 적용"""
    p = points.copy()
    p[:, 1] = -p[:, 1]
    p[:, 2] = -p[:, 2]
    return p

os.makedirs("/content/outputs", exist_ok=True)

colors_rgba = np.hstack([all_cols, np.full((all_cols.shape[0], 1), 255, dtype=np.uint8)])
raw_pcd = trimesh.PointCloud(vertices=to_opengl(all_pts), colors=colors_rgba)
raw_scene = trimesh.Scene()
raw_scene.add_geometry(raw_pcd, geom_name="dust3r_raw")

glb_path = f"/content/outputs/dust3r_pointcloud_{N_FRAMES}_raw.glb"
raw_scene.export(glb_path)

# unlit 처리 (VGGT 때와 동일 — Babylon Sandbox 등에서 밝게 뜨는 문제 방지)
gltf = GLTF2().load(glb_path)
for material in gltf.materials:
    if material.extensions is None:
        material.extensions = {}
    material.extensions["KHR_materials_unlit"] = {}
    if material.pbrMetallicRoughness:
        material.pbrMetallicRoughness.metallicFactor = 0.0
        material.pbrMetallicRoughness.roughnessFactor = 1.0
if gltf.extensionsUsed is None:
    gltf.extensionsUsed = []
if "KHR_materials_unlit" not in gltf.extensionsUsed:
    gltf.extensionsUsed.append("KHR_materials_unlit")

unlit_path = f"/content/outputs/dust3r_pointcloud_{N_FRAMES}.glb"
gltf.save(unlit_path)
print(f"{unlit_path} 저장 완료 ({len(all_pts):,} points)")

/content/outputs/dust3r_pointcloud_30.glb 저장 완료 (5,849,021 points)
